# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [3]:
# TODO
df['revenue'] = df['qty'] * df['price']
print("Total revenue: ", df['revenue'].sum())
print("Total units: ", df['qty'].sum())

Total revenue:  8520.0
Total units:  783


I added the column 'revenue' which multiplies the units sold by the price, for each row. 8520 is the total revenue, the sum of all the values in the revenue column. Total units sums the values in the 'qty' column to show the total quantity of items sold.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [7]:
# TODO
grouped_df = df.groupby('category').sum().sort_values('revenue', ascending=False)
grouped_df['revenue_share (%)'] = grouped_df['revenue'] / grouped_df['revenue'].sum() * 100
grouped_df = grouped_df.drop(columns=['vendor_id', 'price'])
grouped_df

,qty,revenue,revenue_share (%)
category,,,
Food,362,4293.0,50.387324
Merch,158,1771.5,20.792254
Drink,178,1554.0,18.239437
RainGear,85,901.5,10.580986


I created a new table that groups df by category and sums the row values. I added a new column for revenue share that expresses each group's revenue as a percentage of the total revenue. I also dropped the columns 'vendor_id' and 'price' since adding these across rows does not make sense. Food has the highest revenue (4293 dollars) and revenue share (50.39%) , followed by Merch, Drink, and RainGear.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [8]:
# TODO
df_averages = df.groupby('vendor_id').agg({'revenue': 'mean', 'qty': 'sum'})
df_averages = df_averages.sort_values('revenue', ascending=False)
df_averages

,revenue,qty
vendor_id,,
V-01,22.595745,188
V-18,21.750000,217
V-05,20.580645,178
V-10,20.314286,200


I made a new table that groups df by vendor and computes the average of the revenue column and the sum of the quantity column. Vendor 01 has the highest average revenue (22.60 dollars) on 188 items sold, while Vendor 10 has the lowest average revenue (20.31 dollars) on 200 items sold.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [10]:
# TODO
merch_share = (df[df['category'] == 'Merch']['revenue'].sum()) / (df['revenue'].sum()) * 100
merch_share.round(1)

np.float64(20.8)

I filtered to Merch only and summed its revenue column, and then divided this by the sum of total revenue and multiplied by 100 to find Merch's percent of total revenue. Merch sales have 20.8% of total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [12]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

vendors_merged = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one').fillna({'vendor_name': 'Unknown Vendor'})
print("Row count: ", len(vendors_merged))
print("Total revenue: ", vendors_merged['revenue'].sum())
print("Unmatched vendor: ", vendors_merged[vendors_merged['vendor_name'] == 'Unknown Vendor']['vendor_id'].unique())

Row count:  400
Total revenue:  8520.0
Unmatched vendor:  ['V-18']


The row count is 400 and total revenue is 8520 dollars, which is the same as the original table.

**The unmatched vendor, and what I did about it:** The unmatched vendor is Vendor 18. I assigned it the name "Unknown Vendor" because this vendor accounted for the most items sold of all vendors (217) so it is too significant to ignore in the table.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [15]:
# TODO
vendors_pivot = pd.pivot_table(data=vendors_merged, index='vendor_name', columns='category', values='revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Total')
vendors_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [17]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(grouped_df['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(vendors_merged) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

I would tell the vendors to focus more on their specific area of product. Currently, each vendor sells each type of product to some degree. For example, while Rotunda Tacos makes 882 dollars of revenue on Food sales, it also makes 489 dollars on Merch sales. This is a significant amount, and it does not make much sense because people most likely come to the taco vendor for food, so the vendor can probably be more successful by allocating more of their labor towards food rather than also trying to sell merch. Another good example is Cav Merch North. Their revenue on Food (1054.5 dollars) is more than twice their revenue on Merch (400.5 dollars). However, they also have the lowest average order revenue (20.31 dollars) out of all vendors. If they were to focus on Merch, as their name suggests, they may be more successful in increasing their revenue per order.

My answer to Question 5 is the least trustworthy because I am still skeptical of treating the unknown vendor as simply one vendor with a missing name. This vendor (V-18) has the most items sold of all vendors, and it seems strange that its name would be unknown. It is possible that V-18 could represent something like a group of vendors, or be a placeholder for all other vendors. However, I still think the best course of action is to give it a name and include it in the table.